# 01 - Data Collection

### Objective

Inspect and validate cached NBA datasets used to build the player archetype profiles.

The project uses NBA base, advanced, tracking, shot-location, hustle, and matchup data to capture playing style behavior.

### Goals

1. Document the data sources.
2. Define the eligible player population.
3. Validate dataset structure.
4. Evaluate missing data and player coverage.
5. Identify data limitations before feature engineering.

No modeling or final feature construction is performed here.

### Data Collection

NBA API requests are implemented in 'scripts/collect_data.py'

Raw responses are cached in 'data/raw/2025_26' as Parquet files. (Year changes depending on SEASON variable set in 'src/nba_archetypes/config.py')

This notebook loads the cached datasets rather than repeatedly requesting data from the NBA API.

### Inputs

- Cached NBA API datasets

### Outputs

- Eligible players dataset
- Dataset inventory
- Missing data report
- Column inventory

In [19]:
import pandas as pd
import duckdb

from nba_archetypes.config import SEASON, SEASON_TYPE, RAW_DATA_DIR, INTERIM_DATA_DIR, METADATA_DIR, SQL_DIR, MIN_MINUTES

import nba_archetypes.data_collection as dc

import nba_archetypes.data_validation as dv

### 1. Data Sources and Collection

Player statistics were collected using the Python 'nba_api' package.

The collection pipeline retrieves several categories of data:
- Traditional and advanced player statistics
- Usage data and scoring splits
- Offensive tracking data
- Shot-location data
- Hustle and defensive tracking data
- Defensive player matchups

API requests and caching are managed through 'scripts/collect_data.py'.

Previously collected CSV datasets were migrated to Parquet where necessary (as part of a project restructure). Newly collected datasets are cached directly as Parquet.

This allows the analysis to run without repeatedly making external API requests.

In [15]:
datasets = {}
for file in RAW_DATA_DIR.glob("*.parquet"):
    dataset_name = file.stem
    datasets[dataset_name] = pd.read_parquet(file)

player_types = ["base", "advanced", "scoring", "usage"]

tracking_types = [
    "passing", "rebounding", "speeddistance",
    "possessions", "drives", "catchshoot",
    "pullupshot", "posttouch", "painttouch",
    "efficiency"
]

expected_files = {f"{name}_stats.parquet" for name in player_types + tracking_types}
expected_files.update(["shot_locations.parquet", "hustle_stats.parquet", "matchup_stats.parquet"])

existing_files = {file.name for file in RAW_DATA_DIR.glob("*.parquet")}
missing_files = sorted(expected_files - existing_files)

print("Expected files:", len(expected_files))
print("Files found:", len(existing_files))
print("Missing expected files:", missing_files)

Expected files: 17
Files found: 17
Missing expected files: []


### 2. Data Validation

A validation check is important to make sure our datasets have the required columns needed to identify players and a unique PLAYER_ID to eventually join the tables together properly.

The table below will mark True if the validation check is satisfied. The matchup_stats shows "Not applicable" under the unique_player_id column which is completely normal and does not mean that the dataset cannot be used. It simply means that the data has a different grain than what we are traditionally looking for.

Datasets on a different grain can be engineered later to satisfy our initial requirements.

In [4]:
data_val_report = []
for dataset_name, df in datasets.items():
    if dataset_name == "matchup_stats":
        unique_id = "Not applicable"
        required_cols = dv.check_required_cols(df, ["OFF_PLAYER_ID", "DEF_PLAYER_ID", "PARTIAL_POSS"])
    else:
        unique_id = dv.check_unique_ids(df, "PLAYER_ID")
        required_cols = dv.check_required_cols(df, ["PLAYER_ID", "PLAYER_NAME"])
    
    data_val_report.append({
        "dataset": dataset_name,
        "unique_player_id": unique_id,
        "required_columns": required_cols
    })

data_val_report = pd.DataFrame(data_val_report)
display(data_val_report)

,dataset,unique_player_id,required_columns
0,advanced_stats,True,True
1,base_stats,True,True
2,catchshoot_stats,True,True
3,drives_stats,True,True
4,efficiency_stats,True,True
5,hustle_stats,True,True
6,matchup_stats,Not applicable,True
7,painttouch_stats,True,True
8,passing_stats,True,True
9,possessions_stats,True,True


### 3. Dataset Inventory

Before cleaning or merging the datasets, let's examine the available source tables.

The inventory records the number of rows and columns in each dataset and helps identify missing or imcomplete sources.

In [10]:
inventory = dv.dataset_inventory(datasets)

display(inventory)

,dataset,rows,columns,missing_values
0,advanced_stats,582,79,0
1,base_stats,582,69,0
2,catchshoot_stats,582,16,140
3,drives_stats,582,25,0
4,efficiency_stats,582,22,0
5,hustle_stats,581,28,0
6,matchup_stats,147805,27,0
7,painttouch_stats,582,26,0
8,passing_stats,582,18,0
9,possessions_stats,582,21,0


### 4. Missing Value Analysis

The dataset inventory report above reveals a few datasets containing missing values. We will investigate these a bit further in a later notebook but for now, let's document which columns contain missing values and the missing percentage for each.

Missing values at this stage aren't too alarming yet because they could be from columns that we will not use for our analysis at all or due to low activity which may be eliminated with our eligibility threshold.

We won't make any assumptions until we narrow down our player universe and feature set.

In [11]:
mv_report = dv.missing_value_report(datasets)

display(mv_report)

,dataset,column,missing_percentage
0,catchshoot_stats,CATCH_SHOOT_FG3_PCT,6.19
1,catchshoot_stats,CATCH_SHOOT_FG3M,6.01
2,catchshoot_stats,CATCH_SHOOT_FG3A,6.01
3,catchshoot_stats,CATCH_SHOOT_FG_PCT,2.92
4,catchshoot_stats,CATCH_SHOOT_EFG_PCT,2.92
5,pullupshot_stats,PULL_UP_FG3_PCT,17.18
6,pullupshot_stats,PULL_UP_FG_PCT,7.90
7,pullupshot_stats,PULL_UP_EFG_PCT,7.90
8,pullupshot_stats,PULL_UP_FG3M,6.01
9,pullupshot_stats,PULL_UP_FG3A,6.01


### 5. Player Universe

#### Eligibility Threshold

This project includes players with at least 250 regular season minutes.

This threshold is intended to reduce the influence of very small sample size players while retaining a resonably broad pool of players to work with.

The threshold will be evaluated for sensitivity later in the modeling process.

#### Core Player Data

Our base and advanced stats tables will be used to establish the core player population and provide context such as minutes, games played, and possessions.

These datasets will also provide the identifiers needed to combine our other data sources.

In [14]:
print("Minutes threshold:", MIN_MINUTES)

con = duckdb.connect()

core_data = {
    "base_stats": (RAW_DATA_DIR / "base_stats.parquet").as_posix(),
    "advanced_stats": (RAW_DATA_DIR / "advanced_stats.parquet").as_posix()
}

for view_name, file_path in core_data.items():
    con.execute(f"""
    CREATE OR REPLACE VIEW {view_name} AS
    SELECT *
    FROM '{file_path}'
    """)

sql_query = (SQL_DIR / "01_eligible_players.sql").read_text()

eligible_players = con.execute(sql_query, {"MIN_MINUTES": MIN_MINUTES}).df()

print("Total Players:", len(datasets["base_stats"]))
print("Eligible Players:", len(eligible_players))
print("Unique player IDs:", eligible_players["PLAYER_ID"].is_unique)
print("Missing possessions:", eligible_players["POSS"].isna().sum())
print("Non-positive possessions:", (eligible_players["POSS"] <= 0).sum())

display(eligible_players.head())


Minutes threshold: 250
Total Players: 582
Eligible Players: 450
Unique player IDs: True
Missing possessions: 0
Non-positive possessions: 0


,PLAYER_ID,PLAYER_NAME,TEAM_ABBREVIATION,AGE,GP,MIN,POSS
0,1631260,AJ Green,MIL,26.0,78,2269.933333,4704
1,1642358,AJ Johnson,DAL,21.0,48,453.433333,968
2,203932,Aaron Gordon,DEN,30.0,36,1004.958333,2096
3,1628988,Aaron Holiday,HOU,29.0,57,780.731667,1618
4,1630174,Aaron Nesmith,IND,26.0,45,1334.950000,2871


### 6. Shot Location Source Verification

The shot location nba_api endpoint is formatted differently than the rest of our datasets and requires column rearrangement to make it suitable for our analysis.

This step checks that the cached shot location dataset contains the expected shooting zone columns and that it covers our eligible player population.

In [17]:
shot_locations = datasets["shot_locations"]
shot_columns = [
    "RA_FGA",
    "PAINT_NON_RA_FGA",
    "MIDRANGE_FGA",
    "LEFT_CORNER_3_FGA",
    "RIGHT_CORNER_3_FGA",
    "ABOVE_BREAK_3_FGA",
    "BACKCOURT_FGA",
]

missing_columns = [
    col for col in shot_columns
    if col not in shot_locations.columns
]

eligible_ids = set(eligible_players["PLAYER_ID"])
shot_location_ids = set(shot_locations["PLAYER_ID"])

print("Shot-location rows:", len(shot_locations))
print("Duplicate player IDs:", shot_locations["PLAYER_ID"].duplicated().sum())
print("Missing shot-location columns:", missing_columns)
print("Eligible players missing:", len(eligible_ids - shot_location_ids))
print("All shot-location columns:", shot_locations.columns.tolist())

Shot-location rows: 582
Duplicate player IDs: 0
Missing shot-location columns: []
Eligible players missing: 0
All shot-location columns: ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'NICKNAME', 'RA_FGM', 'RA_FGA', 'RA_FG_PCT', 'PAINT_NON_RA_FGM', 'PAINT_NON_RA_FGA', 'PAINT_NON_RA_FG_PCT', 'MIDRANGE_FGM', 'MIDRANGE_FGA', 'MIDRANGE_FG_PCT', 'LEFT_CORNER_3_FGM', 'LEFT_CORNER_3_FGA', 'LEFT_CORNER_3_FG_PCT', 'RIGHT_CORNER_3_FGM', 'RIGHT_CORNER_3_FGA', 'RIGHT_CORNER_3_FG_PCT', 'ABOVE_BREAK_3_FGM', 'ABOVE_BREAK_3_FGA', 'ABOVE_BREAK_3_FG_PCT', 'BACKCOURT_FGM', 'BACKCOURT_FGA', 'BACKCOURT_FG_PCT', 'CORNER_3_FGM', 'CORNER_3_FGA', 'CORNER_3_PCT']


### 7. Player Data Coverage

Not every NBA dataset is guaranteed to include every eligible player. I measured the percentage of eligible players represented in each data source before deciding whether those variables should be included in our final model.

The matchup_stats dataset has intentionally been left out of this report because an offensive player can matchup up against multiple defenders and a defender against multiple offensive players in that dataset.

In [8]:
unique_player_datasets = {
    name: df 
    for name, df in datasets.items() 
    if name != "matchup_stats"
    }

dv.player_coverage_report(unique_player_datasets, eligible_players)

,dataset,coverage_percentage
0,advanced_stats,100.0
1,base_stats,100.0
2,catchshoot_stats,100.0
3,drives_stats,100.0
4,efficiency_stats,100.0
5,hustle_stats,100.0
6,painttouch_stats,100.0
7,passing_stats,100.0
8,possessions_stats,100.0
9,posttouch_stats,100.0


### 8. Matchup Data

As mentioned earlier, our matchup data is on a different granularity than the rest of our datasets. Unlike the player-level datasets, matchup records contain multiple rows per offensive/defensive player because they can matchup against many different opponents throughout the course of the season.

The objective of this section is to inspect matchup coverage by possessions and ensure the source has sufficient observations for analysis.

Matchup data will eventually be used to support engineered metrics such as defensive versatility, defensive matchup difficulty, and potentially more.

In [8]:
eligible_data = eligible_players.set_index("PLAYER_ID")
defender_poss = datasets["matchup_stats"].groupby("DEF_PLAYER_ID")["PARTIAL_POSS"].sum()

matchup_coverage = (defender_poss / eligible_data["POSS"]).describe()
display(matchup_coverage)

count    450.000000
mean       0.925925
std        0.019144
min        0.843886
25%        0.915140
50%        0.926677
75%        0.939496
max        0.964807
dtype: float64

In [20]:
manifest = []

for file in sorted(RAW_DATA_DIR.glob("*.parquet")):

    name = file.stem

    if name in [f"{x}_stats" for x in player_types]:
        endpoint = "LeagueDashPlayerStats"

    elif name in [f"{x}_stats" for x in tracking_types]:
        endpoint = "LeagueDashPtStats"

    elif name == "shot_locations":
        endpoint = "LeagueDashPlayerShotLocations"

    elif name == "hustle_stats":
        endpoint = "LeagueHustleStatsPlayer"

    elif name == "matchup_stats":
        endpoint = "LeagueSeasonMatchups"

    else:
        endpoint = "Unknown"

    manifest.append({
        "file": file.name,
        "expected_endpoint": endpoint,
        "expected_season": SEASON,
        "expected_season_type": SEASON_TYPE,
        "rows": len(datasets[name]),
        "columns": len(datasets[name].columns),
        "historical_settings_verified": "No",
    })

source_manifest = pd.DataFrame(manifest)

display(source_manifest)

,file,expected_endpoint,expected_season,expected_season_type,rows,columns,historical_settings_verified
0,advanced_stats.parquet,LeagueDashPlayerStats,2025-26,Regular Season,582,79,No
1,base_stats.parquet,LeagueDashPlayerStats,2025-26,Regular Season,582,69,No
2,catchshoot_stats.parquet,LeagueDashPtStats,2025-26,Regular Season,582,16,No
3,drives_stats.parquet,LeagueDashPtStats,2025-26,Regular Season,582,25,No
4,efficiency_stats.parquet,LeagueDashPtStats,2025-26,Regular Season,582,22,No
5,hustle_stats.parquet,LeagueHustleStatsPlayer,2025-26,Regular Season,581,28,No
6,matchup_stats.parquet,LeagueSeasonMatchups,2025-26,Regular Season,147805,27,No
7,painttouch_stats.parquet,LeagueDashPtStats,2025-26,Regular Season,582,26,No
8,passing_stats.parquet,LeagueDashPtStats,2025-26,Regular Season,582,18,No
9,possessions_stats.parquet,LeagueDashPtStats,2025-26,Regular Season,582,21,No


### 9. Data Collection Outputs

The dataset inventory and missing data report are saved to the metadata directory. I am also creating a column inventory to document the variables captured from our data sources. The eligible players DataFrame will be saved to our interim data directory which will be used in later notebooks.

These files document the data sources and variables available for subsequent feature engineering and support reproducibility.

In [12]:
dc.save_parquet_file(eligible_players, "eligible_players", INTERIM_DATA_DIR, overwrite=True)

dc.save_csv_file(inventory, "dataset_inventory", METADATA_DIR, overwrite=True)

dc.save_csv_file(mv_report, "missing_value_report", METADATA_DIR, overwrite=True)

column_inventory = []

for name, df in datasets.items():
    for column in df.columns:
        column_inventory.append({
            "dataset": name,
            "column": column
        })
        
column_inventory = pd.DataFrame(column_inventory)
dc.save_csv_file(column_inventory, "column_inventory", METADATA_DIR, overwrite=True)

Saved eligible_players: 450 rows and 7 columns
Saved dataset_inventory: 17 rows and 4 columns
Saved missing_value_report: 19 rows and 3 columns
Saved column_inventory: 559 rows and 2 columns


WindowsPath('C:/Users/aabre/Documents/data-science-projects/nba-player-archetype-analysis/data/metadata/column_inventory.csv')

### 10. Conclusion

#### Key Findings

- **Data Validation:** All of the datasets are on a player-season grain level except for the matchup_data which is on a defender-opponent-season level.

- **Missing Value Analysis:** Only one column had a missing percentage of higher than 10% which was the PULL_UP_FG3_PCT stat from the pullupshot_stats dataset which had a missing rate of around 17%. There were several other columns with missing percentages ranging from around 8% to less than 1%. These will be investigated further once we narrow down our eligible players and feature set.

- **Player Universe:** 450 players met the eligibility threshold of 250 total minutes played.

- **Player Data Coverage:** There was a 100% coverage rate among all the datasets collected.

- **Matchup Data:** The matchup data has an average coverage rate of 92.6% when comparing matchup possessions to our core advanced_data possessions. The minimum amount of coverage was 84.4% and the maximum coverage was 96.5%. This is a slight limitation for this data source but I believe that the 92.6% average will be suitable for our analysis.

#### Decisions

- Retain all datasets for feature selection/engineering.

- Investigate missing values in catchshoot_stats, pullupshot_stats, and shot_locations before final inclusion.

- Keep matchup data and engineer it to mirror the granularity of the other datasets.

- Sensitivity testing may be needed for the eligibility threshold to ensure it doesn't alter our model significantly.